In [ ]:
import sys

sys.path.append('.')
from piece_v2 import piece_v2

piece_v2.start(should_send_to_score=False)

In [2]:
from soundmining_tools import note, spectrum

fundamental = note.note_to_hertz("C0")
first_partial = note.note_to_hertz("C1")
fact = spectrum.make_fact(fundamental, first_partial)
basic_spectrum = spectrum.make_spectrum(fundamental, fact, 50)
print(list(enumerate(basic_spectrum)))

fm_facts1 = (spectrum.make_fact(basic_spectrum[6], basic_spectrum[11]), spectrum.make_fact(basic_spectrum[11], basic_spectrum[18]))
print(fm_facts1)
fm_facts2 = (spectrum.make_fact(basic_spectrum[5], basic_spectrum[13]), spectrum.make_fact(basic_spectrum[13], basic_spectrum[22]))
print(fm_facts2)

fm_spectrum1 = spectrum.make_fm_synthesis(basic_spectrum[6], fm_facts1[0] * basic_spectrum[6], 50)
fm_spectrum2 = spectrum.make_fm_synthesis(basic_spectrum[5], fm_facts2[0] * basic_spectrum[5], 50)
print(fm_spectrum1)
print(fm_spectrum2)


[(0, 16.3516), (1, 32.7032), (2, 49.0548), (3, 65.4064), (4, 81.75800000000001), (5, 98.1096), (6, 114.4612), (7, 130.8128), (8, 147.1644), (9, 163.51600000000002), (10, 179.8676), (11, 196.2192), (12, 212.57080000000002), (13, 228.9224), (14, 245.27400000000003), (15, 261.6256), (16, 277.97720000000004), (17, 294.3288), (18, 310.6804), (19, 327.03200000000004), (20, 343.3836), (21, 359.7352), (22, 376.08680000000004), (23, 392.4384), (24, 408.79), (25, 425.14160000000004), (26, 441.49320000000006), (27, 457.8448), (28, 474.19640000000004), (29, 490.54800000000006), (30, 506.8996), (31, 523.2512), (32, 539.6028), (33, 555.9544000000001), (34, 572.306), (35, 588.6576), (36, 605.0092000000001), (37, 621.3608), (38, 637.7124), (39, 654.0640000000001), (40, 670.4156), (41, 686.7672), (42, 703.1188000000001), (43, 719.4704), (44, 735.822), (45, 752.1736000000001), (46, 768.5252), (47, 784.8768), (48, 801.2284000000001), (49, 817.58)]
(0.7142857142857142, 0.5833333333333335)
(1.3333333333333

In [ ]:
from soundmining_tools.supercollider_receiver import ExtendedNoteHandler, PatchArguments
from soundmining_tools.supercollider_client import *
from soundmining_tools.supercollider_client import SupercolliderClient
from soundmining_tools.modular.instrument import AddAction, AudioInstrument
from soundmining_tools.generative import *
from soundmining_tools.sequencer import SequenceNote
from soundmining_tools.ui.ui_piece import UiPieceBuilder, UiPiece
from piece_v2 import *
from soundmining_tools.spectrum import make_fm_synthesis, make_fact
from soundmining_tools.note import note_to_hertz
from enum import StrEnum
from dataclasses import dataclass
from typing import Optional
import math
from soundmining_tools.sieve import Sieve, SimpleSieve, UnionSieve, IntersectionSieve
from collections.abc import Callable
from ipywidgets import Output
from ipycanvas import Canvas, hold_canvas
import ipywidgets as widgets
from soundmining_tools.modular.instrument import NodeId

piece_v2.reset()

piece_v2.synth_player.start()

static_control = piece_v2.instruments.static_control
sine_control = piece_v2.instruments.sine_control
perc_control = piece_v2.instruments.perc_control
line_control = piece_v2.instruments.line_control
xline_control = piece_v2.instruments.xline_control
signal_sum = piece_v2.instruments.signal_sum
signal_multiply = piece_v2.instruments.signal_multiply
three_block_control = piece_v2.instruments.three_block_control

def low_modindex(amp: float) -> float:
    return 10 + (random_range(5, 20) * amp)
    
def high_modindex(amp: float) -> float:
    return 100 + (random_range(500, 2000) * amp)

FM1_TRACK = "FM1"
SHORT_FM1_TRACK = "SHORT_FM1"
FM2_TRACK = "FM2"
SHORT_FM2_TRACK = "SHORT_FM2"
RESONATORS1_TRACK = "RESONATORS1"
RESONATORS1_MIRROR_TRACK = "RESONATORS1_MIRROR"
SHORT_RESONATORS1_TRACK = "SHORT_RESONATORS1"
SHORT_RESONATORS1_MIRROR_TRACK = "SHORT_RESONATORS1_MIRROR"
RESONATORS2_TRACK = "RESONATORS2"
RESONATORS2_TRACK_MIRROR = "RESONATORS2_MIRROR"
SHORT_RESONATORS2_TRACK = "SHORT_RESONATORS2"
SHORT_RESONATORS2_MIRROR_TRACK = "SHORT_RESONATORS2_MIRROR"

OUTPUT = {
    FM1_TRACK: 0,
    SHORT_FM1_TRACK: 2,
    FM2_TRACK: 4,
    SHORT_FM2_TRACK: 6,
    RESONATORS1_TRACK: 8,
    RESONATORS1_MIRROR_TRACK: 10,
    SHORT_RESONATORS1_TRACK: 12,
    SHORT_RESONATORS1_MIRROR_TRACK: 14,
    RESONATORS2_TRACK: 16,
    RESONATORS2_TRACK_MIRROR: 18,
    SHORT_RESONATORS2_TRACK: 20,
    SHORT_RESONATORS2_MIRROR_TRACK: 22,
}

def play_fm1(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black") -> SequenceNote:
    mod_freq = static_control(pitch * mod_ratio1)        
    mod_amp = static_control(low_modindex(amp))        
    mod = (
        piece_v2.synth_player.note()
        .sine(freq=mod_freq, amp=mod_amp)
        .audio_stack.pop()
    )        
    car_freq = static_control(pitch)
    car_amp = sine_control(0, amp)    
    fm_mod = signal_sum(car_freq, mod).add_action(AddAction.TAIL_ACTION)
    (
        piece_v2.synth_player.note()
            .sine(freq=fm_mod, amp=car_amp)
            .pan(pan_control)
            .play(start, duration, output_bus=OUTPUT[FM1_TRACK])
    )
    return SequenceNote(start=start, track=FM1_TRACK, duration=duration, freq=pitch, color=note_color)

def play_short_fm1(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black") -> SequenceNote:
    mod_freq = static_control(pitch * mod_ratio1)        
    mod_amp = static_control(low_modindex(amp))        
    mod = (
        piece_v2.synth_player.note()
        .sine(freq=mod_freq, amp=mod_amp)
        .audio_stack.pop()
    )        
    car_freq = static_control(pitch)    
    car_amp = xline_control(amp, 0.000001)    
    fm_mod = signal_sum(car_freq, mod).add_action(AddAction.TAIL_ACTION)
    (
        piece_v2.synth_player.note()
            .sine(freq=fm_mod, amp=car_amp)
            .pan(pan_control)
            .play(start, duration, output_bus=OUTPUT[SHORT_FM1_TRACK])
    )
    return SequenceNote(start=start, track=SHORT_FM1_TRACK, duration=duration, freq=pitch, color=note_color)
    
    
def play_fm2(start: float, duration: float, pitch: float, mod_ratio1: float, mod_ratio2: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    amp = amp * 0.2
    mod_freq1 = static_control(pitch * mod_ratio1)        
    mod_amp1 = static_control(low_modindex(amp))                
    mod1 = (
        piece_v2.synth_player.note()
        .saw(freq=mod_freq1, amp=mod_amp1)
        .audio_stack.pop()
    )

    mod_freq2 = static_control(pitch * mod_ratio2)      
    mod_index2 = static_control(low_modindex(amp))     # Turn to noise with larger values        
    mod_amp2 = signal_multiply(mod_freq2, mod_index2).add_action(AddAction.TAIL_ACTION)
    fm_mod2 = signal_sum(mod_freq2, mod1).add_action(AddAction.TAIL_ACTION)
    mod2 = (
        piece_v2.synth_player.note()
        .saw(freq=fm_mod2, amp=mod_amp2)
        .audio_stack.pop()
    )
    car_freq = static_control(pitch)
    car_amp = sine_control(0, amp)
    fm_mod = signal_sum(car_freq, mod2).add_action(AddAction.TAIL_ACTION)
    (
        piece_v2.synth_player.note()
            .saw(freq=fm_mod, amp=car_amp)
            .pan(pan_control)
            .play(start, duration, output_bus=OUTPUT[FM2_TRACK])
    )
    return SequenceNote(start=start, track=FM2_TRACK, duration=duration, freq=pitch, color=note_color)

def play_short_fm2(start: float, duration: float, pitch: float, mod_ratio1: float, mod_ratio2: float, amp: float, pan_control: AudioInstrument, note_color: str = "black") -> SequenceNote:
    amp = amp * 0.2
    mod_freq1 = static_control(pitch * mod_ratio1)        
    mod_amp1 = static_control(low_modindex(amp))                
    mod1 = (
        piece_v2.synth_player.note()
        .triangle(freq=mod_freq1, amp=mod_amp1)
        .audio_stack.pop()
    )

    mod_freq2 = static_control(pitch * mod_ratio2)      
    mod_index2 = static_control(low_modindex(amp))     # Turn to noise with larger values        
    mod_amp2 = signal_multiply(mod_freq2, mod_index2).add_action(AddAction.TAIL_ACTION)
    fm_mod2 = signal_sum(mod_freq2, mod1).add_action(AddAction.TAIL_ACTION)
    mod2 = (
        piece_v2.synth_player.note()
        .triangle(freq=fm_mod2, amp=mod_amp2)
        .audio_stack.pop()
    )
    car_freq = static_control(pitch)
    car_amp = xline_control(amp, 0.000001)

    fm_mod = signal_sum(car_freq, mod2).add_action(AddAction.TAIL_ACTION)
    (
        piece_v2.synth_player.note()
            .triangle(freq=fm_mod, amp=car_amp)
            .pan(pan_control)
            .play(start, duration, output_bus=OUTPUT[SHORT_FM2_TRACK])
    )
    return SequenceNote(start=start, track=SHORT_FM2_TRACK, duration=duration, freq=pitch, color=note_color)


def amp_func1(i: int) -> float:
    return 1 / ((i + 1) * math.pi)

def amp_func2(i: int) -> float:
    return 4.0 / math.pow((i + 1.0) * math.pi, 2)

def resonator_amp(i: int, amp_func: Callable[[int], float], sieve: Sieve):
    return amp_func(i) if sieve.is_sieve(i) else 0

def play_resonators2(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 2)])
    
    scaled_amp = amp * 1
    volume_control = sine_control(0.0, scaled_amp)    

    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[0] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    phases = [0 for i in range(15)]
    (
        piece_v2.synth_player.note()
        .bank_of_osc(freqs, amps, phases)
        .mono_volume(volume_control)
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[RESONATORS2_TRACK])
    )
    return SequenceNote(start=start, track=RESONATORS2_TRACK, duration=duration, freq=pitch, color=note_color)

def play_short_resonators2(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 2)])
    
    scaled_amp = amp * 1
    volume_control = xline_control(scaled_amp, 0.000001)    

    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[0] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    phases = [0 for i in range(15)]
    (
        piece_v2.synth_player.note()
        .bank_of_osc(freqs, amps, phases)
        .mono_volume(volume_control)
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[SHORT_RESONATORS2_TRACK])
    )
    return SequenceNote(start=start, track=SHORT_RESONATORS2_TRACK, duration=duration, freq=pitch, color=note_color)

def play_resonators2_mirror(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 2)])

    scaled_amp = amp * 1
    volume_control = sine_control(0.0, scaled_amp)    
    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[1] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    phases = [0 for i in range(15)]
    (
        piece_v2.synth_player.note()
        .bank_of_osc(freqs, amps, phases)
        .mono_volume(volume_control)
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[RESONATORS2_TRACK_MIRROR])
    )
    return SequenceNote(start=start, track=RESONATORS2_TRACK_MIRROR, duration=duration, freq=pitch, color=note_color)

def play_short_resonators2_mirror(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 2)])

    scaled_amp = amp * 1
    volume_control = xline_control(scaled_amp, 0.000001)    
    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[1] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    phases = [0 for i in range(15)]
    (
        piece_v2.synth_player.note()
        .bank_of_osc(freqs, amps, phases)
        .mono_volume(volume_control)
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[SHORT_RESONATORS2_MIRROR_TRACK])
    )
    return SequenceNote(start=start, track=SHORT_RESONATORS1_MIRROR_TRACK, duration=duration, freq=pitch, color=note_color)

def play_resonators1(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 1)])

    scaled_amp = amp * 0.2
    volume_control = sine_control(0.0, scaled_amp)    
    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[0] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    ring_times = [random_range(0.3, 0.5) for i in range(15)]
    (
        piece_v2.synth_player.note()
        .white_noise(volume_control)
        .bank_of_resonators(freqs, amps, ring_times)        
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[RESONATORS1_TRACK])
    )
    return SequenceNote(start=start, track=RESONATORS1_TRACK, duration=duration, freq=pitch, color=note_color)

def play_short_resonators1(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 1)])

    scaled_amp = amp * 0.4
    volume_control = xline_control(scaled_amp, 0.000001)    
    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[0] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    ring_times = [random_range(0.3, 0.5) for i in range(15)]
    (
        piece_v2.synth_player.note()
        .white_noise(static_control(amp))
        .bank_of_resonators(freqs, amps, ring_times)  
        .mono_volume(volume_control)      
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[SHORT_RESONATORS1_TRACK])
    )
    return SequenceNote(start=start, track=SHORT_RESONATORS1_TRACK, duration=duration, freq=pitch, color=note_color)

def play_resonators1_mirror(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 1)])

    scaled_amp = amp * 0.2
    volume_control = sine_control(0.0, scaled_amp)    
    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[1] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    ring_times = [random_range(0.3, 0.5) for i in range(15)]
    (
        piece_v2.synth_player.note()
        .white_noise(volume_control)
        .bank_of_resonators(freqs, amps, ring_times)        
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[RESONATORS1_MIRROR_TRACK])
    )
    return SequenceNote(start=start, track=RESONATORS1_MIRROR_TRACK, duration=duration, freq=pitch, color=note_color)

def play_short_resonators1_mirror(start: float, duration: float, pitch: float, mod_ratio1: float, amp: float, pan_control: AudioInstrument, note_color: str = "black"):
    sieve = UnionSieve([SimpleSieve(4, 0), SimpleSieve(3, 1)])

    scaled_amp = amp * 0.4
    volume_control = xline_control(scaled_amp, 0.000001)    
    fm_spectrum = spectrum.make_fm_synthesis(pitch, pitch * mod_ratio1, 50)
    freqs = [partial[1] for partial in fm_spectrum]
    amps = [resonator_amp(i, amp_func1, sieve) for i in range(15)]
    ring_times = [random_range(0.3, 0.5) for i in range(15)]
    (
        piece_v2.synth_player.note()
        .white_noise(static_control(amp))
        .bank_of_resonators(freqs, amps, ring_times)   
        .mono_volume(volume_control)     
        .pan(pan_control)
        .play(start, duration, output_bus=OUTPUT[SHORT_RESONATORS1_MIRROR_TRACK])
    )
    return SequenceNote(start=start, track=SHORT_RESONATORS1_MIRROR_TRACK, duration=duration, freq=pitch, color=note_color)


In [7]:
PieceInstrument = StrEnum("PieceInstrument", ["FM1", "FM2", "RESONATOR1", "RESONATOR1_MIRROR", "RESONATOR2", "RESONATOR2_MIRROR"])

class Part1:

    long_pad_instruments_chain = MarkovChain({
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.0, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.3, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.0, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.0,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2_MIRROR,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.0},
    }, (PieceInstrument.FM1,))

    long_extra_pad_instruments_chain = MarkovChain({
        (PieceInstrument.FM2,): {(PieceInstrument.FM2, ): 0.4, (PieceInstrument.RESONATOR1,): 0.3, (PieceInstrument.RESONATOR1_MIRROR,): 0.3},
        (PieceInstrument.RESONATOR1,): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.0, (PieceInstrument.RESONATOR1_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR1_MIRROR, ): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.1, (PieceInstrument.RESONATOR1_MIRROR,): 0.0}
    }, (PieceInstrument.FM2,))

    should_long_extra_pad_chain = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.5, False: 0.5}
        }, True
    )

    should_long_do_pan_line = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.33, False: 0.66}
        }, True
    )

    @classmethod
    def make_long_pan(cls, ranges: list[tuple[float, float]]) -> AudioInstrument:
        if cls.should_long_do_pan_line.next():
            pan_start, pan_end = pan_line(0.5, ranges)        
            return line_control(pan_start, pan_end)
        else:
            pan = pan_point(ranges)        
            return static_control(pan)

    should_long_do_short_duration = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.33, False: 0.66}
        }, True
    )

    @classmethod
    def make_long_start_duration(cls) -> tuple[float, float]:
        if cls.should_long_do_short_duration.next():
            duration = random_range(8, 13)
            start_delay = random_range(3, 5)
        else:
            duration = random_range(13, 21)
            start_delay = 0    
        return (start_delay, duration)

    @classmethod
    def play_long_instrument1(cls, start: float, pitch: float, facts: tuple[float, float], amp: float, note_color: str) -> list[SequenceNote]:
        notes = []
        piece_instrument_choise = cls.long_pad_instruments_chain.next()    
        for piece_instrument in piece_instrument_choise:        
            start_delay, duration = cls.make_long_start_duration()
            match piece_instrument:
                case PieceInstrument.FM1:                                
                    pan_control = cls.make_long_pan([(-0.25, 0.25)])                
                    notes.append(play_fm1(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2:                
                    pan_control = cls.make_long_pan([(-0.50, -0.25), (0.25, 0.50)])
                    notes.append(play_resonators2(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2_MIRROR:                
                    pan_control = cls.make_long_pan([(-0.50, -0.25), (0.25, 0.50)])
                    notes.append(play_resonators2_mirror(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
        if cls.should_long_extra_pad_chain.next():
            piece_extra_instrument_choise = cls.long_extra_pad_instruments_chain.next()                       
            start_delay, duration = cls.make_long_start_duration()
            for extra_piece_instrument in piece_extra_instrument_choise:
                match extra_piece_instrument:
                    case PieceInstrument.FM2:                    
                        pan_control = cls.make_long_pan([(-0.75, -0.50), (0.50, 0.75)])
                        notes.append(play_fm2(start + start_delay, duration, pitch, facts[0], facts[1], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1:                    
                        pan_control = cls.make_long_pan([(-0.99, -0.75), (0.75, 0.99)])
                        notes.append(play_resonators1(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1_MIRROR:                    
                        pan_control = cls.make_long_pan([(-0.99, -0.75), (0.75, 0.99)])
                        notes.append(play_resonators1_mirror(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color)) 
        return notes   


    short_instruments_chain = MarkovChain({
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.0, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.3, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.0, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.0,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2_MIRROR,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.0},
    }, (PieceInstrument.FM1,))

    short_extra_instruments_chain = MarkovChain({
        (PieceInstrument.FM2,): {(PieceInstrument.FM2, ): 0.4, (PieceInstrument.RESONATOR1,): 0.3, (PieceInstrument.RESONATOR1_MIRROR,): 0.3},
        (PieceInstrument.RESONATOR1,): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.0, (PieceInstrument.RESONATOR1_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR1_MIRROR, ): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.1, (PieceInstrument.RESONATOR1_MIRROR,): 0.0}
    }, (PieceInstrument.FM2,))

    should_short_extra_chain = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.5, False: 0.5}
        }, True
    )

    @classmethod
    def play_short_instrument1(cls, start: float, pitch: float, facts: tuple[float, float], amp: float, note_color: str) -> list[SequenceNote]:
        notes = []
        piece_instrument_choise = cls.short_instruments_chain.next()    
        for piece_instrument in piece_instrument_choise:
            start_time = start + random_range(-0.005, 0.005)        
            duration = random_range(0.05, 0.1)
            match piece_instrument:
                case PieceInstrument.FM1:     
                    pan = pan_point([(-0.25, 0.25)])
                    pan_control = static_control(pan)
                    notes.append(play_short_fm1(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2:
                    pan = pan_point([(-0.50, -0.25), (0.25, 0.50)])
                    pan_control = static_control(pan)
                    notes.append(play_short_resonators2(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2_MIRROR:
                    pan = pan_point([(-0.50, -0.25), (0.25, 0.50)])
                    pan_control = static_control(pan)
                    notes.append(play_short_resonators2_mirror(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))    
        if cls.should_short_extra_chain.next():
            start_time = start + random_range(0.01, 0.05)
            piece_extra_instrument_choise = cls.short_extra_instruments_chain.next()  
            duration = random_range(0.05, 0.4)
            for extra_piece_instrument in piece_extra_instrument_choise:
                match extra_piece_instrument:
                    case PieceInstrument.FM2:
                        pan = pan_point([(-0.75, -0.50), (0.50, 0.75)]) 
                        pan_control = static_control(pan)
                        notes.append(play_short_fm2(start_time, duration, pitch, facts[0], facts[1], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1:
                        pan = pan_point([(-0.99, -0.75), (0.75, 0.99)])
                        pan_control = static_control(pan)
                        notes.append(play_short_resonators1(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1_MIRROR:
                        pan = pan_point([(-0.99, -0.75), (0.75, 0.99)])
                        pan_control = static_control(pan)
                        notes.append(play_short_resonators1_mirror(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
        return notes

    @classmethod
    def play_long_melody1(cls, start: float, repeats: int) -> list[SequenceNote]:
        notes = []
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        spectrum_notes = [partial[0] for partial in fm_spectrum]
        mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]
        melody = [0, 2, 1, 3, 6, 4, 5, 7, 10, 9, 8, 11]    
        time = start
        for i in range(repeats):
            for note in melody:
                amp = random_range(0.15, 0.85)
                mirror_amp = random_range(0.15, 0.85)
                notes += cls.play_long_instrument1(time, spectrum_notes[note], fm_facts, amp, note_color="black")
                mirror_time = time + random_range(8, 13)
                notes += cls.play_long_instrument1(mirror_time, mirror_spectrum_notes[note + 1], fm_facts, mirror_amp, note_color="blue")
                time = time + (random_range(13, 21) * random_range(0.33, 0.66))
        return notes

    @classmethod
    def play_short_melody1(cls, start: float, repeats: int) -> list[SequenceNote]:
        notes = []
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        spectrum_notes = [partial[0] for partial in fm_spectrum]
        mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]
        melody = [0, 2, 1, 3, 6, 4, 5, 7, 10, 9, 8, 11]    
        time = start
        mirror_time = time + random_range(4, 7)
        for i in range(repeats):
            for note in melody:
                amp = random_range(0.15, 0.85)   
                mirror_amp = random_range(0.15, 0.85)     
                notes += cls.play_short_instrument1(time, spectrum_notes[note], fm_facts, amp, note_color="black")
                time = time + random_range(3, 5)
                notes += cls.play_short_instrument1(mirror_time, mirror_spectrum_notes[note + 1], fm_facts, mirror_amp, note_color="blue")
                mirror_time = mirror_time + random_range(4, 7)
        return notes


    @classmethod
    def play_short_melody2(cls, start: float, repeats: int) -> list[SequenceNote]:
        notes = []
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        spectrum_notes = [partial[0] for partial in fm_spectrum]
        mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]
        melody = [0, 2, 1, 3, 6, 4, 5, 7, 10, 9, 8, 11]    
        time = start
        mirror_time = time + random_range(3, 4)
        for i in range(repeats):
            for note in melody:
                amp = random_range(0.15, 0.85)   
                mirror_amp = random_range(0.15, 0.85)     
                notes += cls.play_short_instrument1(time, spectrum_notes[note], fm_facts, amp, note_color="black")
                time = time + random_range(3, 5)
                notes += cls.play_short_instrument1(mirror_time, mirror_spectrum_notes[note + 1], fm_facts, mirror_amp, note_color="blue")
                mirror_time = mirror_time + random_range(3, 4)
        return notes
    
    @classmethod
    def get_notes_duration(cls, notes: list[SequenceNote]) -> float:
        duration = 0
        for note in notes:
            duration = max(duration, note.start + note.duration)
        return duration

    @classmethod
    def play_part1(cls, start: float) -> list[SequenceNote]:
        notes = []
        notes += cls.play_long_melody1(start, 2)
        melody1_end = cls.get_notes_duration(notes)
        notes += cls.play_short_melody1(start + random_range(13, 21), 2)
        notes += cls.play_short_melody2(melody1_end - random_range(13, 21), 1)
        return notes

In [5]:
# calculate grids

def get_grid(start: int, step: int, size: int) -> list[int]:
    return [(step * i) + start for i in range(size)]

get_grid(16, 3, 12)

get_grid(13, 3, 12)


[13, 16, 19, 22, 25, 28, 31, 34, 37, 40, 43, 46]

In [23]:
# part 2
class Part2:

    melody = [0, 2, 1, 3, 6, 4, 5, 7, 10, 9, 8, 11]
    mirror_melody = [n + 1 for n in melody]
    fm_facts = fm_facts1
    fm_spectrum = fm_spectrum1
    spectrum_notes = [partial[0] for partial in fm_spectrum]
    mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]

    # High grid
    high_grid = get_grid(16, 3, 12)

    basic_grid = get_grid(0, 1, 12)
    one_grid = get_grid(1, 1, 12)
    """
    Play variants of cells of the theme. Long (1, 2) and short separated. Repeat variant of the same cell. Play different cells 
    in long and short at the same time.

    Short cloud with grid get_grid(16, 3, 12) and long notes (same grid) range 3, 5.

    """
    @classmethod
    def play_long_instrument2(cls, start: float, pitch: float, amp: float, note_color: str = "black") -> list[SequenceNote]:
        notes = []
        start_delay = 0
        piece_instrument_choise = Part1.long_pad_instruments_chain.next()    
        for piece_instrument in piece_instrument_choise:        
            #duration = random_range(1, 2)
            duration = random_range(3, 5)
            match piece_instrument:
                case PieceInstrument.FM1:                                
                    pan_control = Part1.make_long_pan([(-0.25, 0.25)])                
                    notes.append(play_fm1(start + start_delay, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2:                
                    pan_control = Part1.make_long_pan([(-0.50, -0.25), (0.25, 0.50)])
                    notes.append(play_resonators2(start + start_delay, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2_MIRROR:                
                    pan_control = Part1.make_long_pan([(-0.50, -0.25), (0.25, 0.50)])
                    notes.append(play_resonators2_mirror(start + start_delay, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color))
        
        if Part1.should_long_extra_pad_chain.next():
            piece_extra_instrument_choise = Part1.long_extra_pad_instruments_chain.next()                       
            duration = random_range(3, 5)
            for extra_piece_instrument in piece_extra_instrument_choise:
                match extra_piece_instrument:
                    case PieceInstrument.FM2:                    
                        pan_control = Part1.make_long_pan([(-0.75, -0.50), (0.50, 0.75)])
                        notes.append(play_fm2(start + start_delay, duration, pitch, fm_facts1[0], fm_facts1[1], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1:                    
                        pan_control = Part1.make_long_pan([(-0.99, -0.75), (0.75, 0.99)])
                        notes.append(play_resonators1(start + start_delay, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1_MIRROR:                    
                        pan_control = Part1.make_long_pan([(-0.99, -0.75), (0.75, 0.99)])
                        notes.append(play_resonators1_mirror(start + start_delay, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color)) 
        return notes  

    @classmethod
    def play_short_instrument2(cls, start: float, pitch: float, amp: float, note_color: str = "black") -> list[SequenceNote]:
        notes = []
        piece_instrument_choise = Part1.short_instruments_chain.next()    
        for piece_instrument in piece_instrument_choise:
            start_time = start + random_range(-0.005, 0.005)        
            duration = random_range(0.05, 0.1)
            match piece_instrument:
                case PieceInstrument.FM1:     
                    pan = pan_point([(-0.25, 0.25)])
                    pan_control = static_control(pan)
                    notes.append(play_short_fm1(start_time, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2:
                    pan = pan_point([(-0.50, -0.25), (0.25, 0.50)])
                    pan_control = static_control(pan)
                    notes.append(play_short_resonators2(start_time, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2_MIRROR:
                    pan = pan_point([(-0.50, -0.25), (0.25, 0.50)])
                    pan_control = static_control(pan)
                    notes.append(play_short_resonators2_mirror(start_time, duration, pitch, fm_facts1[0], amp, pan_control, note_color=note_color))    
        
        return notes


    short_cell_melody_trans_1 = MarkovChain({
            (7, 10, 9, 8, 11): {(7, 10, 9, 8, 11): 0.33, (11, 8, 9, 10, 7): 0.66},
            (11, 8, 9, 10, 7): {(7, 10, 9, 8, 11): 0.66, (11, 8, 9, 10, 7): 0.33},        
        }, (7, 10, 9, 8, 11)
    )

    short_cell_melody_part_1 = MarkovChain({
        (0, 1, 2, 3, 4): {(0, 1, 2, 3, 4): 0.1, (0, 1, 2, 3): 0.5, (0, 1, 2): 0.3, (1, 2, 3): 0.3},
        (0, 1, 2, 3): {(0, 1, 2, 3, 4): 0.4, (0, 1, 2, 3): 0.1, (0, 1, 2): 0.3, (1, 2, 3): 0.3},
        (0, 1, 2): {(0, 1, 2, 3, 4): 0.4, (0, 1, 2, 3): 0.5, (0, 1, 2): 0.0, (1, 2, 3): 0.1},
        (1, 2, 3): {(0, 1, 2, 3, 4): 0.4, (0, 1, 2, 3): 0.5, (0, 1, 2): 0.1, (1, 2, 3): 0.0},
    }, (0, 1, 2, 3, 4))

    @classmethod
    def play_short_cell_melody1(cls, start: float, notes: list[float], grid: list[int]) -> list[SequenceNote]:
        """
        7 10 9 8 11
        11 8 9 10 7

        01234 0123 012 123
        """
        seq_notes = []
        time = start
        transport = cls.short_cell_melody_trans_1.next()
        for n in cls.short_cell_melody_part_1.next():
            amp = random_range(0.15, 0.85)
            seq_notes += cls.play_short_instrument2(time, notes[grid[transport[n]]], amp)
            time += random_range(0.2, 0.4)
        return seq_notes

    long_cell_melody_trans_1 = MarkovChain({
            (0, 2, 1): {(0, 2, 1): 0.1, (1, 2, 0): 0.3, (2, 0, 1): 0.3, (1, 0, 2): 0.3},
            (1, 2, 0): {(0, 2, 1): 0.3, (1, 2, 0): 0.1, (2, 0, 1): 0.3, (1, 0, 2): 0.3},
            (2, 0, 1): {(0, 2, 1): 0.3, (1, 2, 0): 0.3, (2, 0, 1): 0.1, (1, 0, 2): 0.3},
            (1, 0, 2): {(0, 2, 1): 0.3, (1, 2, 0): 0.3, (2, 0, 1): 0.3, (1, 0, 2): 0.1},
        }, (0, 2, 1)
    )

    long_cell_melody_part_1 = MarkovChain({
            (0, 1, 2): {(0, 1, 2): 0.4, (0, 1): 0.6, (0,): 0.2},
            (0, 1): {(0, 1, 2): 0.5, (0, 1): 0.3, (0,): 0.2},
            (0, ): {(0, 1, 2): 0.6, (0, 1): 0.4, (0,): 0.0}
        }, (0, 1, 2)
    )

    @classmethod
    def play_long_cell_melody1(cls, start: float, notes: list[float], grid: list[int]) -> list[SequenceNote]:
        """
        021 120 201 102

        012 01 0
        """
        seq_notes = []
        time = start
        transport = cls.long_cell_melody_trans_1.next()
        for n in cls.long_cell_melody_part_1.next():
            amp = random_range(0.15, 0.85)
            seq_notes += cls.play_long_instrument2(time, notes[grid[transport[n]]], amp)
            time += random_range(3, 5)
        return seq_notes

    long_cell_melody_trans_2 = MarkovChain({
            (3, 6, 4, 5): {(3, 6, 4, 5): 0.1, (5, 4, 6, 3): 0.3, (6, 3, 5, 4): 0.3, (4, 5, 3, 6): 0.3},
            (5, 4, 6, 3): {(3, 6, 4, 5): 0.3, (5, 4, 6, 3): 0.1, (6, 3, 5, 4): 0.3, (4, 5, 3, 6): 0.3},
            (6, 3, 5, 4): {(3, 6, 4, 5): 0.3, (5, 4, 6, 3): 0.3, (6, 3, 5, 4): 0.1, (4, 5, 3, 6): 0.3},
            (4, 5, 3, 6): {(3, 6, 4, 5): 0.3, (5, 4, 6, 3): 0.3, (6, 3, 5, 4): 0.3, (4, 5, 3, 6): 0.1},
        }, (3, 6, 4, 5)
    )

    long_cell_melody_part_2 = MarkovChain({
        (0, 1, 2, 3): {(0, 1, 2, 3): 0.1, (0, 1, 2): 0.5, (0, 1): 0.3, (0,): 0.3},
        (0, 1, 2): {(0, 1, 2, 3): 0.4, (0, 1, 2): 0.1, (0, 1): 0.3, (0,): 0.3},
        (0, 1): {(0, 1, 2, 3): 0.4, (0, 1, 2): 0.5, (0, 1): 0.0, (0,): 0.1},
        (0,): {(0, 1, 2, 3): 0.4, (0, 1, 2): 0.5, (0, 1): 0.1, (0,): 0.0},
    }, (0, 1, 2, 3))

    @classmethod
    def play_long_cell_melody2(cls, start: float, notes: list[float], grid: list[int]) -> list[SequenceNote]:
        """
        high grid

        3, 6, 4, 5
        5, 4, 6, 3
        6, 3, 5, 4
        4, 5, 3, 6

        0123 012 01 0

        """
        seq_notes = []
        time = start
        transport = cls.long_cell_melody_trans_2.next()
        for n in cls.long_cell_melody_part_2.next():
            amp = random_range(0.15, 0.85)
            seq_notes += cls.play_long_instrument2(time, notes[grid[transport[n]]], amp)
            time += random_range(3, 5)
        return seq_notes

    short_cell_melody_trans_2 = MarkovChain({
            (7, 10, 9, 8, 11): {(7, 10, 9, 8, 11): 0.33, (11, 8, 9, 10, 7): 0.66},
            (11, 8, 9, 10, 7): {(7, 10, 9, 8, 11): 0.66, (11, 8, 9, 10, 7): 0.33},        
        }, (7, 10, 9, 8, 11)
    )

    short_cell_melody_part_2 = MarkovChain({
        (0, 1, 2, 3, 4): {(0, 1, 2, 3, 4): 0.1, (0, 1, 2, 3): 0.5, (0, 1, 2): 0.3, (1, 2, 3): 0.3},
        (0, 1, 2, 3): {(0, 1, 2, 3, 4): 0.4, (0, 1, 2, 3): 0.1, (0, 1, 2): 0.3, (1, 2, 3): 0.3},
        (0, 1, 2): {(0, 1, 2, 3, 4): 0.4, (0, 1, 2, 3): 0.5, (0, 1, 2): 0.0, (1, 2, 3): 0.1},
        (1, 2, 3): {(0, 1, 2, 3, 4): 0.4, (0, 1, 2, 3): 0.5, (0, 1, 2): 0.1, (1, 2, 3): 0.0},
    }, (0, 1, 2, 3, 4))

    @classmethod
    def play_short_cell_melody2(cls, start: float, notes: list[float], grid: list[int], duration: float) -> list[SequenceNote]:
        """
        high grid

        7 10 9 8 11
        11 8 9 10 7

        01234 0123 012 123    

        """
        seq_notes = []
        time = start
        end = start + duration
        while time <= end:    
            transport = cls.short_cell_melody_trans_2.next()
            for n in cls.short_cell_melody_part_2.next():
                amp = random_range(0.15, 0.85)
                seq_notes += cls.play_short_instrument2(time, notes[grid[transport[n]]], amp)
                time += random_range(0.1, 0.9)
        return seq_notes

    """
    13 21 34 55 89 144
    """
    @classmethod
    def play_part2(cls, start: float) -> list[SequenceNote]:
        seq_notes = []
        grid1 = cls.basic_grid
        time1 = start
        time2 = start + random_range(3, 5)
        duration1 = 55
        end1 = start + duration1
        while time1 <= end1:
            melody_notes = cls.play_long_cell_melody1(time1, cls.spectrum_notes, grid1)
            melody_duration = Part1.get_notes_duration(melody_notes)
            time1 = melody_duration + random_range(5, 8)
            seq_notes += melody_notes
        while time2 <= end1:
            short_melody_notes = cls.play_short_cell_melody1(time2, cls.spectrum_notes, grid1)
            short_melody_duration = Part1.get_notes_duration(short_melody_notes)
            time2 = short_melody_duration + random_range(3, 5)
            seq_notes += short_melody_notes

        start3 = max(time1, time2) - random_range(1, 2)
        
        duration2 = 55
        grid2 = cls.high_grid
        seq_notes += cls.play_short_cell_melody2(start3, cls.spectrum_notes, grid2, duration2)
        time3 = start3 + random_range(1, 2)

        end3 = start3 + duration2
        while time3 < end3:
            long_melody_notes2 = cls.play_long_cell_melody2(time3, cls.spectrum_notes, cls.high_grid)
            long_melody_duration = Part1.get_notes_duration(long_melody_notes2)
            time3 = long_melody_duration + random_range(5, 8)
            seq_notes += long_melody_notes2

        grid3 = cls.one_grid
        duration3 = 34
        time4 = time3 - random_range(1, 2)
        time5 = time4 + random_range(1, 2)
        end4 = time4 + duration3

        while time4 <= end4:
            melody_notes = cls.play_long_cell_melody1(time4, cls.mirror_spectrum_notes, grid3)
            melody_duration = Part1.get_notes_duration(melody_notes)
            time4 = melody_duration + random_range(5, 8)
            seq_notes += melody_notes
        while time5 <= end4:
            short_melody_notes = cls.play_short_cell_melody1(time5, cls.mirror_spectrum_notes, grid3)
            short_melody_duration = Part1.get_notes_duration(short_melody_notes)
            time5 = short_melody_duration + random_range(3, 5)
            seq_notes += short_melody_notes


        return seq_notes


class MyHandler2(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient):
        super().__init__(client)
    
    def play_basic_melody(self, patch_arguments: PatchArguments) -> None:        
        match patch_arguments.note:
            case 0:
                notes = spectrum_notes
                grid = basic_grid
            case 1:
                notes = mirror_spectrum_notes
                grid = one_grid
            case _:
                notes = spectrum_notes
                grid = basic_grid
        match patch_arguments.octave:
            case 2:
                play_long_cell_melody1(patch_arguments.start, notes, grid)
            case 3:
                play_short_cell_melody1(patch_arguments.start, notes, grid)
            case 4:
                grid = high_grid
                play_long_cell_melody2(patch_arguments.start, notes, grid)
                #play_long_instrument2(patch_arguments.start, notes[grid[patch_arguments.note]], patch_arguments.amp)
            case 5:
                grid = high_grid
                play_short_cell_melody2(patch_arguments.start, notes, grid, duration=10)
                #play_short_instrument2(patch_arguments.start, notes[grid[patch_arguments.note]], patch_arguments.amp)
    
    def start_part2(self, patch_arguments: PatchArguments):
        Part2.play_part2(patch_arguments.start)
            

    def handle_note(self, patch_arguments):
        return self.start_part2(patch_arguments)

my_handler2 = MyHandler2(piece_v2.supercollider_client)
#piece_v2.receiver.set_note_handler(my_handler2)
    


In [19]:
class Part3:

    long_pad_instruments_chain = MarkovChain({
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.0, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.3, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.3, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.0, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.FM1,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.0, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.0,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR2_MIRROR,): {
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2): 0.2, 
            (PieceInstrument.FM1, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.RESONATOR2, PieceInstrument.RESONATOR2_MIRROR): 0.2, 
            (PieceInstrument.FM1,): 0.1, 
            (PieceInstrument.RESONATOR2,): 0.1,
            (PieceInstrument.RESONATOR2_MIRROR,): 0.0},
    }, (PieceInstrument.FM1,))

    long_extra_pad_instruments_chain = MarkovChain({
        (PieceInstrument.FM2,): {(PieceInstrument.FM2, ): 0.4, (PieceInstrument.RESONATOR1,): 0.3, (PieceInstrument.RESONATOR1_MIRROR,): 0.3},
        (PieceInstrument.RESONATOR1,): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.0, (PieceInstrument.RESONATOR1_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR1_MIRROR, ): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.1, (PieceInstrument.RESONATOR1_MIRROR,): 0.0}
    }, (PieceInstrument.FM2,))

    should_long_extra_pad_chain = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.5, False: 0.5}
        }, True
    )

    should_long_do_pan_line = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.33, False: 0.66}
        }, True
    )

    @classmethod
    def make_long_pan(cls, ranges: list[tuple[float, float]]) -> AudioInstrument:
        if cls.should_long_do_pan_line.next():
            pan_start, pan_end = pan_line(0.5, ranges)        
            return line_control(pan_start, pan_end)
        else:
            pan = pan_point(ranges)        
            return static_control(pan)

    should_long_do_short_duration = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.33, False: 0.66}
        }, True
    )

    @classmethod
    def make_long_start_duration(cls) -> tuple[float, float]:
        if cls.should_long_do_short_duration.next():
            duration = random_range(3, 5)
            start_delay = random_range(1, 2)
        else:
            duration = random_range(5, 8)
            start_delay = 0    
        return (start_delay, duration)

    @classmethod
    def play_long_instrument3(cls, start: float, pitch: float, facts: tuple[float, float], amp: float, note_color: str) -> list[SequenceNote]:
        notes = []
        piece_instrument_choise = cls.long_pad_instruments_chain.next()    
        for piece_instrument in piece_instrument_choise:        
            start_delay, duration = cls.make_long_start_duration()
            match piece_instrument:
                case PieceInstrument.FM1:                                
                    pan_control = cls.make_long_pan([(-0.25, 0.25)])                
                    notes.append(play_fm1(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2:                
                    pan_control = cls.make_long_pan([(-0.50, -0.25), (0.25, 0.50)])
                    notes.append(play_resonators2(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2_MIRROR:                
                    pan_control = cls.make_long_pan([(-0.50, -0.25), (0.25, 0.50)])
                    notes.append(play_resonators2_mirror(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
        if cls.should_long_extra_pad_chain.next():
            piece_extra_instrument_choise = cls.long_extra_pad_instruments_chain.next()                       
            start_delay, duration = cls.make_long_start_duration()
            for extra_piece_instrument in piece_extra_instrument_choise:
                match extra_piece_instrument:
                    case PieceInstrument.FM2:                    
                        pan_control = cls.make_long_pan([(-0.75, -0.50), (0.50, 0.75)])
                        notes.append(play_fm2(start + start_delay, duration, pitch, facts[0], facts[1], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1:                    
                        pan_control = cls.make_long_pan([(-0.99, -0.75), (0.75, 0.99)])
                        notes.append(play_resonators1(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1_MIRROR:                    
                        pan_control = cls.make_long_pan([(-0.99, -0.75), (0.75, 0.99)])
                        notes.append(play_resonators1_mirror(start + start_delay, duration, pitch, facts[0], amp, pan_control, note_color=note_color)) 
        return notes 


    short_extra_instruments_chain = MarkovChain({
        (PieceInstrument.FM2,): {(PieceInstrument.FM2, ): 0.4, (PieceInstrument.RESONATOR1,): 0.3, (PieceInstrument.RESONATOR1_MIRROR,): 0.3},
        (PieceInstrument.RESONATOR1,): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.0, (PieceInstrument.RESONATOR1_MIRROR,): 0.1},
        (PieceInstrument.RESONATOR1_MIRROR, ): {(PieceInstrument.FM2, ): 0.9, (PieceInstrument.RESONATOR1,): 0.1, (PieceInstrument.RESONATOR1_MIRROR,): 0.0}
    }, (PieceInstrument.FM2,))

    should_short_extra_chain = MarkovChain(
        {
            True: {True: 0, False: 1},
            False: {True: 0.2, False: 0.8}
        }, True
    )

    @classmethod
    def play_short_instrument3(cls, start: float, pitch: float, facts: tuple[float, float], amp: float, note_color: str = "black") -> list[SequenceNote]:
        notes = []
        piece_instrument_choise = Part1.short_instruments_chain.next()    
        for piece_instrument in piece_instrument_choise:
            start_time = start + random_range(-0.005, 0.005)        
            duration = random_range(0.05, 0.1)
            match piece_instrument:
                case PieceInstrument.FM1:     
                    pan = pan_point([(-0.25, 0.25)])
                    pan_control = static_control(pan)
                    notes.append(play_short_fm1(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2:
                    pan = pan_point([(-0.50, -0.25), (0.25, 0.50)])
                    pan_control = static_control(pan)
                    notes.append(play_short_resonators2(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                case PieceInstrument.RESONATOR2_MIRROR:
                    pan = pan_point([(-0.50, -0.25), (0.25, 0.50)])
                    pan_control = static_control(pan)
                    notes.append(play_short_resonators2_mirror(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))  

        if Part3.should_short_extra_chain.next():
            start_time = start + random_range(0.01, 0.05)
            piece_extra_instrument_choise = Part3.short_extra_instruments_chain.next()  
            duration = random_range(0.05, 0.4)
            for extra_piece_instrument in piece_extra_instrument_choise:
                match extra_piece_instrument:
                    case PieceInstrument.FM2:
                        pan = pan_point([(-0.75, -0.50), (0.50, 0.75)]) 
                        pan_control = static_control(pan)
                        notes.append(play_short_fm2(start_time, duration, pitch, facts[0], facts[1], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1:
                        pan = pan_point([(-0.99, -0.75), (0.75, 0.99)])
                        pan_control = static_control(pan)
                        notes.append(play_short_resonators1(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))
                    case PieceInstrument.RESONATOR1_MIRROR:
                        pan = pan_point([(-0.99, -0.75), (0.75, 0.99)])
                        pan_control = static_control(pan)
                        notes.append(play_short_resonators1_mirror(start_time, duration, pitch, facts[0], amp, pan_control, note_color=note_color))  
        
        return notes


    """
    Orignal melody
    0, 2, 1, 
    3, 6, 4, 5, 
    7, 10, 9, 8, 11

    Variants group 1
    0, 2, 1 
    1, 2, 0 
    2, 0, 1 
    1, 0, 2

    Variants group 2
    3, 6, 4, 5
    5, 4, 6, 3
    6, 3, 5, 4
    4, 5, 3, 6

    Variants group 3
    7 10 9 8 11
    11 8 9 10 7
    """


    melody_cell1_trans = MarkovChain({
            (0, 2, 1): {(0, 2, 1): 0.1, (1, 2, 0): 0.3, (2, 0, 1): 0.3, (1, 0, 2): 0.3},
            (1, 2, 0): {(0, 2, 1): 0.3, (1, 2, 0): 0.1, (2, 0, 1): 0.3, (1, 0, 2): 0.3},
            (2, 0, 1): {(0, 2, 1): 0.3, (1, 2, 0): 0.3, (2, 0, 1): 0.1, (1, 0, 2): 0.3},
            (1, 0, 2): {(0, 2, 1): 0.3, (1, 2, 0): 0.3, (2, 0, 1): 0.3, (1, 0, 2): 0.1},
        }, (0, 2, 1)
    )

    melody_cell2_trans = MarkovChain({
            (3, 6, 4, 5): {(3, 6, 4, 5): 0.1, (5, 4, 6, 3): 0.3, (6, 3, 5, 4): 0.3, (4, 5, 3, 6): 0.3},
            (5, 4, 6, 3): {(3, 6, 4, 5): 0.3, (5, 4, 6, 3): 0.1, (6, 3, 5, 4): 0.3, (4, 5, 3, 6): 0.3},
            (6, 3, 5, 4): {(3, 6, 4, 5): 0.3, (5, 4, 6, 3): 0.3, (6, 3, 5, 4): 0.1, (4, 5, 3, 6): 0.3},
            (4, 5, 3, 6): {(3, 6, 4, 5): 0.3, (5, 4, 6, 3): 0.3, (6, 3, 5, 4): 0.3, (4, 5, 3, 6): 0.1},
        }, (3, 6, 4, 5)
    )

    melody_cell3_trans = MarkovChain({
            (7, 10, 9, 8, 11): {(7, 10, 9, 8, 11): 0.33, (11, 8, 9, 10, 7): 0.66},
            (11, 8, 9, 10, 7): {(7, 10, 9, 8, 11): 0.66, (11, 8, 9, 10, 7): 0.33},        
        }, (7, 10, 9, 8, 11)
    )

    high_grid = get_grid(16, 3, 12)
    high_mirror_grid = get_grid(13, 3, 12)
    basic_grid = get_grid(0, 1, 12)
    one_grid = get_grid(1, 1, 12)

    print(high_grid)
    
    @classmethod
    def long_melody1(cls, start: float, grid: list[int], mirror_grid: list[int]) -> list[SequenceNote]:
        notes = []
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        spectrum_notes = [partial[0] for partial in fm_spectrum]
        mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]        
        melody = list(cls.melody_cell1_trans.next()) + list(cls.melody_cell2_trans.next()) + list(cls.melody_cell3_trans.next())
        time = start        
        for note in melody:
            amp = random_range(0.15, 0.85)
            mirror_amp = random_range(0.15, 0.85)                    
            notes += cls.play_long_instrument3(time, spectrum_notes[grid[note]], fm_facts, amp, note_color="black")
            mirror_time = time + random_range(5, 8)
            notes += cls.play_long_instrument3(mirror_time, mirror_spectrum_notes[mirror_grid[note]], fm_facts, mirror_amp, note_color="blue")
            time = time + (random_range(5, 8) * random_range(0.33, 0.66))
        return notes
    
    @classmethod
    def long_melody1_retro(cls, start: float, grid: list[int], mirror_grid: list[int]) -> list[SequenceNote]:
        notes = []
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        spectrum_notes = [partial[0] for partial in fm_spectrum]
        mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]        
        melody = list(cls.melody_cell3_trans.next()) + list(cls.melody_cell2_trans.next()) + list(cls.melody_cell1_trans.next())
        time = start        
        for note in melody:
            amp = random_range(0.15, 0.85)
            mirror_amp = random_range(0.15, 0.85)
            notes += cls.play_long_instrument3(time, spectrum_notes[grid[note]], fm_facts, amp, note_color="black")
            mirror_time = time + random_range(5, 8)
            notes += cls.play_long_instrument3(mirror_time, mirror_spectrum_notes[mirror_grid[note]], fm_facts, mirror_amp, note_color="blue")
            time = time + (random_range(5, 8) * random_range(0.33, 0.66))
        return notes
    

    should_short_do_short_duration = MarkovChain(
        {
            True: {True: 0.33, False: 0.66},
            False: {True: 0.5, False: 0.5}
        }, True
    )

    @classmethod
    def make_short_duration(cls) -> float:
        if cls.should_short_do_short_duration.next():
            return random_range(0.1, 0.3)
        else:
            return random_range(0.9, 1.5)

    @classmethod
    def short_melody1(cls, start: float, grid: list[int], mirror_grid: list[int]) -> list[SequenceNote]:
        notes = []
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        spectrum_notes = [partial[0] for partial in fm_spectrum]
        mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]
        melody = list(cls.melody_cell1_trans.next()) + list(cls.melody_cell2_trans.next()) + list(cls.melody_cell3_trans.next())
        time = start        
        for note in melody:
            amp = random_range(0.15, 0.85)
            mirror_amp = random_range(0.15, 0.85)                
            notes += cls.play_short_instrument3(time, spectrum_notes[grid[note]], fm_facts, amp, note_color="black")                
            mirror_time = time + cls.make_short_duration()
            notes += cls.play_short_instrument3(mirror_time, mirror_spectrum_notes[mirror_grid[note]], fm_facts, mirror_amp, note_color="blue")
            time = time + (cls.make_short_duration() * random_range(0.33, 0.66))
        return notes
    
    @classmethod
    def short_melody1_retro(cls, start: float, grid: list[int], mirror_grid: list[int]) -> list[SequenceNote]:
        notes = []
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        spectrum_notes = [partial[0] for partial in fm_spectrum]
        mirror_spectrum_notes = [partial[1] for partial in fm_spectrum]
        melody = list(cls.melody_cell3_trans.next()) + list(cls.melody_cell2_trans.next()) + list(cls.melody_cell1_trans.next())
        time = start        
        for note in melody:
            amp = random_range(0.15, 0.85)
            mirror_amp = random_range(0.15, 0.85)
            notes += cls.play_short_instrument3(time, spectrum_notes[grid[note]], fm_facts, amp, note_color="black")                
            mirror_time = time + cls.make_short_duration()
            notes += cls.play_short_instrument3(mirror_time, mirror_spectrum_notes[mirror_grid[note]], fm_facts, mirror_amp, note_color="blue")
            time = time + (cls.make_short_duration() * random_range(0.33, 0.66))
        return notes
    
    @classmethod
    def play_part3(cls, start: float) -> list[SequenceNote]:
        notes = []
        time = start
        
        grid = get_grid(0, 1, 12)
        mirror_grid = get_grid(1, 1, 12)
        notes += Part3.long_melody1(time, grid, mirror_grid)
        
        time += random_range(13, 21)
        grid = get_grid(2, 2, 12)
        mirror_grid = get_grid(3, 2, 12)
        notes += Part3.short_melody1(time, grid, mirror_grid)

        time += random_range(5, 8)
        grid = get_grid(5, 3, 12)
        mirror_grid = get_grid(7, 3, 12)
        notes += Part3.long_melody1(time, grid, mirror_grid)

        time += random_range(13, 21)
        grid = get_grid(16, 3, 12)
        mirror_grid = get_grid(13, 3, 12)
        notes += Part3.short_melody1(time, grid, mirror_grid)

        time += random_range(5, 8)
        grid = get_grid(16, 3, 12)
        mirror_grid = get_grid(13, 3, 12)
        notes += Part3.long_melody1_retro(time, grid, mirror_grid)

        time += random_range(13, 21)
        grid = get_grid(5, 3, 12)
        mirror_grid = get_grid(7, 3, 12)
        notes += Part3.short_melody1_retro(time, grid, mirror_grid)

        time += random_range(5, 8)
        grid = get_grid(2, 2, 12)
        mirror_grid = get_grid(3, 2, 12)
        notes += Part3.long_melody1_retro(time, grid, mirror_grid)

        time += random_range(13, 21)      
        grid = get_grid(0, 1, 12)
        mirror_grid = get_grid(1, 1, 12)
        notes += Part3.short_melody1_retro(time, grid, mirror_grid)
        return notes
        

    
class MyHandler3(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient):
        super().__init__(client)
    
    
    def play_long_melody1(self, patch_arguments: PatchArguments):
        match patch_arguments.note:
            case 0:
                Part3.long_melody1(patch_arguments.start, Part3.basic_grid, Part3.one_grid)
            case 1:
                Part3.long_melody1_retro(patch_arguments.start, Part3.basic_grid, Part3.one_grid)
            case 2:
                Part3.short_melody1(patch_arguments.start, Part3.basic_grid, Part3.one_grid)
            case 3:
                Part3.short_melody1_retro(patch_arguments.start, Part3.basic_grid, Part3.one_grid)
            case 4:
                Part3.play_part3(patch_arguments.start)
            

    def play_grids1(self, patch_arguments: PatchArguments):        
        match patch_arguments.note:
            case 0:
                grid = get_grid(0, 1, 12)
                mirror_grid = get_grid(1, 1, 12)
            case 1:
                grid = get_grid(2, 2, 12)
                mirror_grid = get_grid(3, 2, 12)
            case 2:
                grid = get_grid(5, 3, 12)
                mirror_grid = get_grid(7, 3, 12)
            case 3:
                grid = get_grid(16, 3, 12)
                mirror_grid = get_grid(13, 3, 12)

        if grid and mirror_grid:
            print(grid, mirror_grid)
            Part3.long_melody1(patch_arguments.start, grid, mirror_grid)
            #Part3.short_melody1(patch_arguments.start, grid, mirror_grid)

    def handle_note(self, patch_arguments):
        #return self.play_long_melody1(patch_arguments)
        #return self.play_grids1(patch_arguments)
        Part3.play_part3(patch_arguments.start)

my_handler3 = MyHandler3(piece_v2.supercollider_client)
piece_v2.receiver.set_note_handler(my_handler3)

[16, 19, 22, 25, 28, 31, 34, 37, 40, 43, 46, 49]


In [ ]:
piece_v2.reset()

if piece_v2.synth_player.should_send_to_score:
    score = piece_v2.synth_player.supercollider_score
    display(score)
    score.add_message(supercollider_client.group_head(0, instrument.NodeId.SOURCE.value))
    score.add_message(supercollider_client.group_tail(NodeId.SOURCE.value, NodeId.EFFECT.value))
    score.add_message(supercollider_client.group_tail(NodeId.EFFECT.value, NodeId.ROOM_EFFECT.value))
    score.add_message(supercollider_client.load_dir(V2_SYNTH_DIR))    


piece_v2.synth_player.start()
notes = Part1.play_part1(0)
#notes = Part2.play_part2(0)
#notes = Part3.play_part3(0)
#notes = []

if piece_v2.synth_player.should_send_to_score:
    piece_v2.synth_player.supercollider_score.make_score_file("module-music-11-part1-v1.txt")
    #piece_v2.synth_player.supercollider_score.make_score_file("module-music-11-part2-v1.txt")
    #piece_v2.synth_player.supercollider_score.make_score_file("module-music-11-part3-v1.txt")    

ui_piece = UiPieceBuilder().add_notes(notes).build()
piece_duration = ui_piece.get_duration()
piece_stats = {"total": piece_duration, "total minutes": piece_duration / 60.0, "tracks": len(ui_piece.tracks)}

min_freq = 0
max_freq = 0
for track in ui_piece.tracks:
    track_duration = 0
    for note in track.notes:
        track_duration = max(track_duration, note.start + note.duration)
        min_freq = min(min_freq, note.freq)
        max_freq = max(max_freq, note.freq)

    piece_stats[track.track_name] = track_duration
display(piece_stats)

TRACK_HEIGHT = 100
NOTE_SCALE_FACTOR = 5
HEIGHT_INDENT = 80

ui_width = 200 + (ui_piece.get_duration() * NOTE_SCALE_FACTOR)
ui_height = TRACK_HEIGHT * len(ui_piece.tracks)

canvas = Canvas(width=ui_width, height=ui_height)

out = Output()

@out.capture()
def handle_mouse_down(x, y):
    canvas.flush()
    print("Mouse down event:", x, y)

canvas.on_mouse_down(handle_mouse_down)
canvas.global_alpha = 0.7

display(canvas)

with hold_canvas():
    canvas.clear()
    for track_index, track in enumerate(sorted(ui_piece.tracks, key=lambda tr: tr.track_name)):
        canvas.font = "14px sans-serif"
        canvas.fill_style = "Black"
        canvas.fill_text(
            track.track_name, x=20, y=(track_index * TRACK_HEIGHT) + HEIGHT_INDENT
        )
        canvas.stroke_style = "Black"
        canvas.stroke_lines(
            [
                (150, (track_index * TRACK_HEIGHT) + 10),
                (150, ((track_index * TRACK_HEIGHT) + TRACK_HEIGHT - 10)),
            ]
        )
        for note in track.notes:
            relative_note = (note.freq - min_freq) / (max_freq - min_freq)
            startx = 200 + (note.start * NOTE_SCALE_FACTOR)
            starty = (
                (track_index * TRACK_HEIGHT)
                - (relative_note * HEIGHT_INDENT)
                + HEIGHT_INDENT
            )
            peakx = 200 + (note.start + (note.duration * note.peak)) * NOTE_SCALE_FACTOR
            peaky = (
                (track_index * TRACK_HEIGHT)
                - (relative_note * HEIGHT_INDENT)
                + HEIGHT_INDENT
                - 5
            )
            endx = 200 + (note.start + note.duration) * NOTE_SCALE_FACTOR
            endy = (
                (track_index * TRACK_HEIGHT)
                - (relative_note * HEIGHT_INDENT)
                + HEIGHT_INDENT
            )
            canvas.stroke_style = note.color
            canvas.stroke_lines([(startx, starty), (peakx, peaky), (endx, endy)])

stop_button = widgets.Button(description="Stop")
status = widgets.Output()
display(stop_button, status)
with status:
    print("Playing")

def stop_playback(b):
    piece_v2.reset()
    canvas.clear()
    status.clear_output()
    with status:
        print("Playback stopped")


stop_button.on_click(stop_playback)

{'total': 116.67512601991,
 'total minutes': 1.9445854336651667,
 'tracks': 11,
 'FM1': 115.00599951566579,
 'RESONATORS2': 108.52151999615563,
 'FM2': 108.488164450382,
 'RESONATORS2_MIRROR': 116.67512601991,
 'RESONATORS1_MIRROR': 83.46174338850035,
 'RESONATORS1': 100.76357254297712,
 'SHORT_FM1': 90.66577859037332,
 'SHORT_RESONATORS2': 91.26618045221379,
 'SHORT_RESONATORS1_MIRROR': 92.14989521994781,
 'SHORT_RESONATORS1': 90.88281482254912,
 'SHORT_FM2': 90.15850070993754}

Canvas(height=1100, width=783)

Button(description='Stop', style=ButtonStyle())

Output()

In [ ]:
class MyHandler(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient):
        super().__init__(client)

    # 6, 11, 18, 
    # 5, 13, 22
    def play_basic_spectrum(self, patch_arguments: PatchArguments) -> None:
        partial = patch_arguments.midi_note - 36
        partial_pitch = basic_spectrum[partial]
        print(f"{partial} {partial_pitch}")
        (
            piece_v2.synth_player.note()
            .sine(static_control(partial_pitch), sine_control(0, 0.5))
            .pan(static_control(random_range(-0.5, 0.5)))
            .play(patch_arguments.start, 3)
        )

    # 1:1, 2:1, 3:1
    # 0:0, 1:0, 2:0
    # 
    # 4, 5, 6, 7:1
    # 3, 4, 5, 6:0 

    # 8 9 10 11 12: 1
    # 7 8 9 10 11: 0

    # 0 2 1  3 6 4 5   7 10 9 8 11  : 0

    def handle_fm(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        duration = random_range(5, 8)
        pan = random_range(-0.99, 0.99)
        pan_control = static_control(pan)
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                
                play_fm1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_fm1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)
            case 4:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_fm2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], fm_facts[1], patch_arguments.amp, pan_control)
            case 5:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_fm2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], fm_facts[1], patch_arguments.amp, pan_control)


    def handle_short_fm(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        duration = random_range(0.05, 0.1)
        pan = random_range(-0.99, 0.99)
        pan_control = static_control(pan)
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_short_fm1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_short_fm1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)
            case 4:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_short_fm2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], fm_facts[1], patch_arguments.amp, pan_control)
            case 5:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_short_fm2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], fm_facts[1], patch_arguments.amp, pan_control)

    def handle_resonators(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1        
        duration = random_range(5, 8)
        pan = random_range(-0.99, 0.99)
        pan_control = static_control(pan)
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_resonators2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)                
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_resonators2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)     
            case 4:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_resonators1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)           
            case 5:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_resonators1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)           

    def handle_short_resonators(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1        
        duration = random_range(0.05, 0.1)
        pan = random_range(-0.99, 0.99)
        pan_control = static_control(pan)
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_short_resonators2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)                
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_short_resonators2(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)     
            case 4:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_short_resonators1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)           
            case 5:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_short_resonators1(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)  

    def handle_resonators_mirror(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1        
        duration = random_range(5, 8)
        pan = random_range(-0.99, 0.99)
        pan_control = static_control(pan)
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_resonators2_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)                
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_resonators2_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)                
            case 4:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_resonators1_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)           
            case 5:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_resonators1_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)  

    def handle_short_resonators_mirror(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1        
        duration = random_range(0.05, 0.1)
        pan = random_range(-0.99, 0.99)
        pan_control = static_control(pan)
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_short_resonators2_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)                
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_short_resonators2_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)                
            case 4:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
                play_short_resonators1_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)           
            case 5:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
                play_short_resonators1_mirror(patch_arguments.start, duration, spectrum_notes[patch_arguments.note], fm_facts[0], patch_arguments.amp, pan_control)           


    def handle_long_instruments1(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1        
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
            
        play_long_instrument1(patch_arguments.start, spectrum_notes[patch_arguments.note], fm_facts, patch_arguments.amp)

    def handle_short_instruments1(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1        
        match patch_arguments.octave:
            case 2:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
            case 3:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
            
        play_short_instrument1(patch_arguments.start, spectrum_notes[patch_arguments.note], fm_facts, patch_arguments.amp)

    def handle_instruments1(self, patch_arguments: PatchArguments):
        fm_facts = fm_facts1
        fm_spectrum = fm_spectrum1
        match patch_arguments.octave:
            case 2 | 4:
                spectrum_notes = [partial[0] for partial in fm_spectrum]
            case 3 | 5:
                spectrum_notes = [partial[1] for partial in fm_spectrum]
        
        match patch_arguments.octave:
            case 2 | 3:
                play_long_instrument1(patch_arguments.start, spectrum_notes[patch_arguments.note], fm_facts, patch_arguments.amp)
            case 4 | 5:
                play_short_instrument1(patch_arguments.start, spectrum_notes[patch_arguments.note], fm_facts, patch_arguments.amp)

    def handle_long_melody1(self, patch_arguments: PatchArguments):
        match patch_arguments.note:
            case 0:
                play_long_melody1(patch_arguments.start)
            case 1:
                play_short_melody1(patch_arguments.start)
            case _:
                pass

    def handle_note(self, patch_arguments):
         return self.handle_long_melody1(patch_arguments)

#my_handler = MyHandler(piece_v2.supercollider_client)
#piece_v2.receiver.set_note_handler(my_handler)


In [1]:
piece_v2.stop()

NameError: name 'piece_v2' is not defined